# EDA

## Dataset ABSA 

In [1]:
from pathlib import Path
import pandas as pd
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

DATA_DIR = Path("/Users/mac/Desktop/NLP/E-Commerce/data/raw")
PROC_DIR = Path("/Users/mac/Desktop/NLP/E-Commerce/data/processed")
PROC_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
KEEP_COLS = ["sentence_id", "text", "aspect_term",
             "polarity", "char_from", "char_to", "category", "domain"]

POLARITY_LABELS = {"positive": 0, "negative": 1, "neutral": 2}

In [3]:
# Extract category from XML
def get_sentence_categories(xml_path):
    """
    Returns dict: sentence_id (int) -> primary category (str)
    Uses first listed category per sentence.
    """
    tree = ET.parse(xml_path)
    result = {}
    for sent in tree.getroot().findall("sentence"):
        sid = int(sent.get("id"))
        cat_node = sent.find("aspectCategories")
        if cat_node is not None:
            cats = [c.get("category") for c in cat_node.findall("aspectCategory")]
            result[sid] = cats[0] if cats else None
    return result

In [4]:
# Load CSV + merge category from XML


def load_domain(csv_path, xml_path, domain):

    df = pd.read_csv(csv_path)
    df.columns = ["sentence_id", "text", "aspect_term",
                  "polarity", "char_from", "char_to"]

    # Attach category
    cats = get_sentence_categories(xml_path)
    df["category"] = df["sentence_id"].map(cats)
    df["domain"]   = domain

    print(f"\n{'='*50}")
    print(f"Domain: {domain.upper()}")
    print(f"{'='*50}")
    print(f"Raw rows        : {len(df)}")
    print(f"Unique sentences: {df['sentence_id'].nunique()}")
    print(f"\nPolarity counts (before filter):")
    print(df["polarity"].value_counts().to_string())

    # Filter
    df = df[
        df["aspect_term"].notna() &        # must have aspect term
        (df["polarity"] != "conflict")     # drop conflict label
    ].copy().reset_index(drop=True)

    print(f"\nAfter filter    : {len(df)} rows")
    print(f"Polarity counts (after filter):")
    print(df["polarity"].value_counts().to_string())

    # Encode label
    df["label"] = df["polarity"].map(POLARITY_LABELS)

    return df[KEEP_COLS + ["label"]]


rest = load_domain(
    DATA_DIR / "Restaurants_Train_v2.csv",
    DATA_DIR / "Restaurants_Train_v2.xml",
    domain="restaurant"
)

lap = load_domain(
    DATA_DIR / "Laptop_Train_v2.csv",
    DATA_DIR / "Laptop_Train_v2.xml",
    domain="laptop"
)


Domain: RESTAURANT
Raw rows        : 3693
Unique sentences: 2021

Polarity counts (before filter):
polarity
positive    2164
negative     805
neutral      633
conflict      91

After filter    : 3602 rows
Polarity counts (after filter):
polarity
positive    2164
negative     805
neutral      633

Domain: LAPTOP
Raw rows        : 2358
Unique sentences: 1488

Polarity counts (before filter):
polarity
positive    987
negative    866
neutral     460
conflict     45

After filter    : 2313 rows
Polarity counts (after filter):
polarity
positive    987
negative    866
neutral     460


In [8]:
# Load trail files as validation set
def load_trial(csv_path, domain):
    """Trial files have no header — assign column names manually."""
    df = pd.read_csv(csv_path, header=None,
                     names=["sentence_id", "text", "aspect_term",
                            "polarity", "char_from", "char_to"])
    df = df[
        df["aspect_term"].notna() &
        (df["polarity"] != "conflict")
    ].copy().reset_index(drop=True)

    df["category"] = None   # trial XML not available after deletion
    df["domain"]   = domain
    df["label"]    = df["polarity"].map(POLARITY_LABELS)
    return df[KEEP_COLS + ["label"]]


rest_val = load_trial(DATA_DIR / "restaurants-trial.csv", "restaurant")
lap_val  = load_trial(DATA_DIR / "laptops-trial.csv",     "laptop")

print(f"\nValidation — Restaurant: {len(rest_val)} rows")
print(f"Validation — Laptop    : {len(lap_val)} rows")


Validation — Restaurant: 97 rows
Validation — Laptop    : 50 rows


In [ ]:
# Train / Test split (split by sentence_id)

from sklearn.model_selection import train_test_split

def sentence_split(df, test_size=0.15, seed=42):
    """
    Split by sentence_id to avoid data leakage.
    (Same sentence must not appear in both train and test.)
    """
    ids = df["sentence_id"].unique()
    train_ids, test_ids = train_test_split(
        ids, test_size=test_size, random_state=seed
    )
    return (df[df["sentence_id"].isin(train_ids)].reset_index(drop=True),
            df[df["sentence_id"].isin(test_ids)].reset_index(drop=True))


rest_train, rest_test = sentence_split(rest)
lap_train,  lap_test  = sentence_split(lap)

print("Final splits:")
print(f"  Restaurant  train={len(rest_train):4}  val={len(rest_val):3}  test={len(rest_test):4}")
print(f"  Laptop      train={len(lap_train):4}  val={len(lap_val):3}  test={len(lap_test):4}")

Final splits:
  Restaurant  train=3031  val= 97  test= 571
  Laptop      train=1946  val= 50  test= 367


In [11]:
# Sanity checks
def sanity_check(df, name):
    print(f"\n[{name}]")
    print(f"  Rows       : {len(df)}")
    print(f"  Null check : {df[KEEP_COLS].isnull().sum().to_dict()}")
    print(f"  Label dist : {df['label'].value_counts().to_dict()}")
    print(f"  Columns    : {df.columns.tolist()}")
    print(f"  Sample row :")
    print(f"    text        : {df.iloc[0]['text']}")
    print(f"    aspect_term : {df.iloc[0]['aspect_term']}")
    print(f"    polarity    : {df.iloc[0]['polarity']}  (label={df.iloc[0]['label']})")
    print(f"    char_from   : {df.iloc[0]['char_from']}  char_to: {df.iloc[0]['char_to']}")
    print(f"    category    : {df.iloc[0]['category']}")

for df, name in [(rest_train, "rest_train"), (rest_val, "rest_val"),
                 (rest_test,  "rest_test"),  (lap_train, "lap_train")]:
    sanity_check(df, name)


[rest_train]
  Rows       : 3031
  Null check : {'sentence_id': 0, 'text': 0, 'aspect_term': 0, 'polarity': 0, 'char_from': 0, 'char_to': 0, 'category': 0, 'domain': 0}
  Label dist : {0: 1818, 1: 674, 2: 539}
  Columns    : ['sentence_id', 'text', 'aspect_term', 'polarity', 'char_from', 'char_to', 'category', 'domain', 'label']
  Sample row :
    text        : But the staff was so horrible to us.
    aspect_term : staff
    polarity    : negative  (label=1)
    char_from   : 8  char_to: 13
    category    : service

[rest_val]
  Rows       : 97
  Null check : {'sentence_id': 0, 'text': 0, 'aspect_term': 0, 'polarity': 0, 'char_from': 0, 'char_to': 0, 'category': 97, 'domain': 0}
  Label dist : {0.0: 68, 1.0: 18, 2.0: 10}
  Columns    : ['sentence_id', 'text', 'aspect_term', 'polarity', 'char_from', 'char_to', 'category', 'domain', 'label']
  Sample row :
    text        : Sentence
    aspect_term : Aspect Term
    polarity    : polarity  (label=nan)
    char_from   : from  char_to: t

In [12]:
# Save processed files
files = {
    "rest_train.csv" : rest_train,
    "rest_val.csv"   : rest_val,
    "rest_test.csv"  : rest_test,
    "lap_train.csv"  : lap_train,
    "lap_val.csv"    : lap_val,
    "lap_test.csv"   : lap_test,
}

for fname, df in files.items():
    path = PROC_DIR / fname
    df.to_csv(path, index=False)
    print(f"Saved: {fname:20} {len(df):4} rows  {path.stat().st_size/1024:.0f} KB")

print("\nAll done. Commit data/processed/ to GitHub.")
print("main.py on Colab only needs: pd.read_csv('data/processed/rest_train.csv')")

Saved: rest_train.csv       3031 rows  434 KB
Saved: rest_val.csv           97 rows  11 KB
Saved: rest_test.csv         571 rows  87 KB
Saved: lap_train.csv        1946 rows  281 KB
Saved: lap_val.csv            50 rows  6 KB
Saved: lap_test.csv          367 rows  52 KB

All done. Commit data/processed/ to GitHub.
main.py on Colab only needs: pd.read_csv('data/processed/rest_train.csv')
